# Notebook 03: Proposed Consistency-Based Training

The proposed method trains the model to keep predictions stable between clean inputs and lightly corrupted inputs. The objective is:

`L = CE(x) + CE(x') + lambda * KL(p(x) || p(x'))`

The KL coefficient is warmed up during early epochs so the network first learns class-discriminative features before the consistency term becomes strong.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import torch
import matplotlib.pyplot as plt

from robust_cifar.data import make_cifar10_loaders, CIFAR10_CLASSES
from robust_cifar.models import build_resnet18_cifar, count_parameters
from robust_cifar.train import get_device, seed_everything

seed_everything(42)
device = get_device()
device

## Consistency Loader

Each training item returns `(clean_image, corrupted_image, label)` using controlled random corruptions: Gaussian noise, blur, JPEG compression, brightness, and contrast.

In [ ]:
train_loader, test_loader = make_cifar10_loaders(
    data_dir=PROJECT_ROOT / "data",
    batch_size=64,
    num_workers=2,
    mode="consistency",
    corruption_severity=2,
    download=True,
)
clean, corrupted, labels = next(iter(train_loader))
clean.shape, corrupted.shape, labels.shape

## Visual Check of Clean and Corrupted Views

In [ ]:
from robust_cifar.visualize import show_batch

show_batch(clean[:16], labels[:16], title="Clean training views")
plt.show()
show_batch(corrupted[:16], labels[:16], title="Corrupted consistency views")
plt.show()

## Train Proposed Model

In [ ]:
from robust_cifar.train import train_consistency

model = build_resnet18_cifar()
EPOCHS = 100
history = train_consistency(
    model,
    train_loader,
    test_loader,
    epochs=EPOCHS,
    lr=0.1,
    lambda_kl=2.0,
    warmup_epochs=10,
    device=device,
    output_dir=PROJECT_ROOT / "checkpoints",
    run_name="resnet18_consistency",
)
metrics_df = (
    pd.DataFrame(history)[["epoch", "train_loss", "train_accuracy", "val_loss", "val_accuracy"]]
    .rename(columns={"val_loss": "test_loss", "val_accuracy": "test_accuracy"})
)

for row in metrics_df.itertuples(index=False):
    print(
        f"Epoch {int(row.epoch):03d} | "
        f"Train Loss: {row.train_loss:.4f} | Train Acc: {row.train_accuracy:.4f} | "
        f"Test Loss: {row.test_loss:.4f} | Test Acc: {row.test_accuracy:.4f}"
    )

metrics_df

## Training and Testing Accuracy Comparison

The table above reports epoch, loss, and accuracy for both training and testing data. The first graph compares training accuracy against testing accuracy; the second graph shows the KL consistency warm-up trend.

In [ ]:
from robust_cifar.visualize import plot_training_history

history_df = pd.DataFrame(history)
plot_training_history(
    history_df,
    title="Consistency ResNet-18",
    output_path=PROJECT_ROOT / "reports" / "figures" / "consistency_training.png",
)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(metrics_df["epoch"], metrics_df["train_accuracy"], label="Training accuracy", linewidth=2)
ax.plot(metrics_df["epoch"], metrics_df["test_accuracy"], label="Testing accuracy", linewidth=2)
ax.set_title("Consistency ResNet-18: Training vs Testing Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "reports" / "figures" / "consistency_train_test_accuracy.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(history_df["epoch"], history_df["kl"], label="KL loss", color="tab:blue")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("KL loss")
ax2 = ax1.twinx()
ax2.plot(history_df["epoch"], history_df["lambda_kl"], label="lambda", color="tab:orange")
ax2.set_ylabel("KL weight")
fig.suptitle("Consistency warm-up behavior")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "reports" / "figures" / "consistency_warmup.png", dpi=180, bbox_inches="tight")
plt.show()